# Featherweight — Phase 2 / Group C: QLoRA training on Colab (T4)

Run **top to bottom** on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).
This is a *thin driver*: all logic lives in the `featherweight` package; this
notebook only orchestrates the Colab-only steps.

> **Caveat (read first):** the GPU-only code below (the Unsloth `train()` call and
> the generation/eval cells) is **authored but Colab-validated** — it follows
> Unsloth's standard Llama-3.1 recipe but exact `SFTConfig` / generation details
> are version-sensitive. The package logic was unit-tested locally.

In [ ]:
!nvidia-smi  # confirm a T4 (Turing, SM 75 -> fp16, not bf16)

## 1. Repo + install

The clone is **idempotent and absolute** so re-running this cell can't nest the
repo inside itself (the cause of earlier path mismatches).

In [ ]:
# Idempotent clone to an ABSOLUTE path, then cd there. Safe to re-run.
![ -d /content/Featherweight/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight.git /content/Featherweight
%cd /content/Featherweight

%pip install -q unsloth mlflow      # Unsloth official install; mlflow for tracking
%pip install -q -e .                # the featherweight package (editable)

> **Editable install + a running kernel:** `pip install -e .` registers the
> package via a `.pth` that Python only reads at startup, so the *current* kernel
> won't see `featherweight` yet. The cell below adds `src/` to the path for this
> session (alternatively: Restart session and re-run from the clone cell).

In [ ]:
import sys

sys.path.insert(0, "/content/Featherweight/src")  # so the running kernel sees the package

import torch, featherweight
from featherweight import config

print("torch CUDA:", torch.cuda.is_available(), "| featherweight", featherweight.__version__)
print("ROOT_DIR:", config.ROOT_DIR)  # MUST be /content/Featherweight (no nesting)

In [4]:
# Optional: capture the resolved GPU stack for reproducibility (commit this file).
!pip freeze > requirements-colab-train.lock.txt

## 2. Workdir, secrets, and the MLflow backend

In [ ]:
from pathlib import Path

# Prefer Google Drive for persistence; fall back to local /content if the mount
# fails (common under the VS Code Colab extension). If you land on local disk,
# /content is ephemeral - download the adapter at the end (last cell).
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/featherweight")
except Exception as e:
    print(f"Drive mount unavailable ({e}); using local /content (ephemeral).")
    DRIVE = Path("/content/featherweight_out")

DRIVE.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = DRIVE / "adapter"
print("workdir:", DRIVE)

In [ ]:
# HF token is OPTIONAL here: the datasets and the unsloth/...bnb-4bit base are all
# ungated. A read token just avoids download rate limits.
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN set - fine (everything used here is ungated).")

In [7]:
# MLflow 3.x errors on the default file:// store (found in Group B), so use sqlite.
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{DRIVE / 'mlflow.db'}")

## 3. Regenerate the formatted data on Colab

Run from the repo root so the data lands in `/content/Featherweight/data`.

In [ ]:
!cd /content/Featherweight && python scripts/prep_data.py

## 4. Smoke run (tiny — pipeline sanity before the full run)

`early_stopping=False` so it doesn't try to evaluate within these few steps.

In [ ]:
from featherweight.train import sft

sft.train(max_steps=10, limit=64, early_stopping=False, output_dir=DRIVE / "adapter_smoke")

## 5. Full QLoRA run

**`max_steps` is a hard ceiling** so the run fits a Colab session (2 full epochs
would be ~16k steps / days on a T4). **Eval-loss early stopping** (on by default)
evaluates a small mixed held-out subset every `config.train.eval_steps` and stops
early once it plateaus, keeping the best checkpoint. Raise `max_steps` if you have
a longer session; tune the budget in Phase 5.

In [ ]:
sft.train(max_steps=500, output_dir=ADAPTER_DIR)

In [ ]:
!ls -lh /content/featherweight_out/adapter

In [ ]:
from huggingface_hub import login, create_repo, upload_folder

login(token="hf_xxx")  # your WRITE token

repo_id = "username/featherweight-adapter"  # change username if different
create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

upload_folder(
    folder_path="/content/featherweight_out/adapter",
    repo_id=repo_id,
    repo_type="model",
)
print("uploaded ->", repo_id)

## 6. Held-out eval: base vs fine-tuned (the Phase 2 verify)

Uses the local-tested `eval.heldout.score`. The generation helper is Colab-validated
(batched left-padded greedy decode); adjust if your stack needs it.

In [ ]:
import torch
from unsloth import FastLanguageModel

from featherweight import config
from featherweight.eval import heldout
from featherweight.utils import tracking

EVAL_N = 200
rows = heldout.load_heldout()[:EVAL_N]


def generate_and_score(model, tokenizer, rows, max_new_tokens=256, batch_size=16):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    preds = []
    for i in range(0, len(rows), batch_size):
        prompts = [r["prompt"] for r in rows[i : i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = out[:, inputs["input_ids"].shape[1] :]
        preds += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return heldout.score(preds, [r["completion"] for r in rows])


base_model, base_tok = FastLanguageModel.from_pretrained(
    config.BASE_MODEL_4BIT,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
base_metrics = generate_and_score(base_model, base_tok, rows)
print("BASE:", base_metrics)

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce fragmentation

import sys

sys.path.insert(0, "/content/Featherweight/src")  # belt-and-suspenders after restart

import torch
from huggingface_hub import login
from unsloth import FastLanguageModel

from featherweight import config
from featherweight.eval import heldout
from featherweight.utils import tracking

EVAL_N = 200
rows = heldout.load_heldout()[:EVAL_N]  # heldout.jsonl survives a restart (disk persists)

# Carried over from the base run so we don't reload the base model:
base_metrics = {
    "n": 200,
    "tool_name_accuracy": 0.835,
    "exact_match_accuracy": 0.305,
    "refusal_accuracy": 0.0,
    "n_refusal": 19,
    "invalid_rate": 0.09,
}


def generate_and_score(model, tokenizer, rows, max_new_tokens=256, batch_size=8):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    preds = []
    for i in range(0, len(rows), batch_size):
        prompts = [r["prompt"] for r in rows[i : i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = out[:, inputs["input_ids"].shape[1] :]
        preds += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return heldout.score(preds, [r["completion"] for r in rows])


login(token="your-huggingface-token")
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    "your-username/featherweight-adapter",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
ft_metrics = generate_and_score(ft_model, ft_tok, rows, batch_size=8)
print("FT:  ", ft_metrics)

with tracking.mlflow_run("heldout-base-vs-ft"):
    tracking.log_metrics({f"base_{k}": v for k, v in base_metrics.items()})
    tracking.log_metrics({f"ft_{k}": v for k, v in ft_metrics.items()})

In [4]:
import mlflow

print("tracking uri:", mlflow.get_tracking_uri())  # the sqlite backend set earlier

runs = mlflow.search_runs(experiment_names=["featherweight"])
metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
# transpose so the many base_/ft_ metrics are readable as rows
print(runs[["tags.mlflow.runName"] + metric_cols].T)

tracking uri: sqlite:////content/mlflow.db
                                                    0
tags.mlflow.runName                heldout-base-vs-ft
metrics.base_n                                  200.0
metrics.ft_tool_name_accuracy                   0.965
metrics.ft_invalid_rate                         0.005
metrics.base_exact_match_accuracy               0.305
metrics.base_refusal_accuracy                     0.0
metrics.base_tool_name_accuracy                 0.835
metrics.ft_exact_match_accuracy                  0.81
metrics.ft_refusal_accuracy                  0.894737
metrics.base_invalid_rate                        0.09
metrics.ft_n                                    200.0
metrics.base_n_refusal                           19.0
metrics.ft_n_refusal                             19.0


## Verify (Phase 2 done when all hold)

- [ ] Full run completed (hit `max_steps` or early-stopped) within the T4 VRAM budget.
- [ ] Training + eval loss decreased (MLflow).
- [ ] `ft_metrics["exact_match_accuracy"]` beats `base_metrics["exact_match_accuracy"]`
      by a clear margin; refusal accuracy improved.
- [ ] Adapter saved (HuggingFace hub).